<a href="https://colab.research.google.com/github/DPrasad612/Movie_recommendation/blob/main/Copy_working_model_for_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q \
streamlit \
pyngrok \
transformers==4.41.2 \
sentence-transformers==2.7.0 \
scikit-learn \
pandas \
numpy \
fer \
mtcnn \
opencv-python-headless \
pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.1/891.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 15.9 MB/s eta 0:00:00


In [3]:
from fer import FER
import cv2
import numpy as np

print("FER Loaded Successfully")

detector = FER(mtcnn=True)
print("MTCNN Loaded Successfully")

ImportError: cannot import name 'FER' from 'fer' (/usr/local/lib/python3.12/dist-packages/fer/__init__.py)

In [2]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -o ml-latest-small.zip

--2026-05-11 04:05:37--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  3.29MB/s    in 0.3s    

2026-05-11 04:05:38 (3.29 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [3]:
# ==============================
# 📦 IMPORTS
# ==============================
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ==============================
# 📊 LOAD DATA
# ==============================
movies = pd.read_csv("ml-latest-small/movies.csv")

print("Dataset Loaded:", movies.shape)

# ==============================
# 🧠 CREATE TEXT FEATURE
# ==============================
movies['overview'] = movies['title'] + " " + movies['genres']

# ==============================
# 🤖 LOAD OR CREATE EMBEDDINGS (FIXED)
# ==============================
if os.path.exists("embeddings.npy"):
    embeddings = np.load("embeddings.npy")
    print("✅ Loaded saved embeddings")
else:
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(
        movies['overview'].tolist(),
        show_progress_bar=True
    )
    np.save("embeddings.npy", embeddings)
    print("✅ Generated & saved embeddings")

# ==============================
# 🔗 COSINE SIMILARITY
# ==============================
cos_sim = cosine_similarity(embeddings)

# ==============================
# 🎯 TEST (OPTIONAL)
# ==============================
def recommend(movie_title, top_n=5):
    idx = movies[movies['title'] == movie_title].index
    if len(idx) == 0:
        return ["Movie not found!"]

    idx = idx[0]
    scores = list(enumerate(cos_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    return [movies.iloc[i[0]]['title'] for i in scores]

print("\n🎬 Test Recommendations:\n")
print(recommend("Toy Story (1995)"))

Dataset Loaded: (9742, 3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

✅ Generated & saved embeddings

🎬 Test Recommendations:

['Toy Story 2 (1999)', 'Goofy Movie, A (1995)', 'Toy Story 3 (2010)', 'Toys (1992)', 'The Lego Movie (2014)']


In [4]:
context = {
    "time": "night",   # try: day / night
    "mood": "happy"    # try: happy / sad / neutral
}
def context_score(row, context):
    score = 0

    # Time-based preference
    if context['time'] == 'night' and 'Horror' in row['genres']:
        score += 2

    # Mood-based preference
    if context['mood'] == 'happy' and 'Comedy' in row['genres']:
        score += 2

    if context['mood'] == 'sad' and 'Drama' in row['genres']:
        score += 2

    return score

def recommend_with_context(movie_title, context, top_n=10):

    idx = movies[movies['title'] == movie_title].index

    if len(idx) == 0:
        return ["Movie not found!"]

    idx = idx[0]

    scores = list(enumerate(cos_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+15]

    results = []

    for i, sim in scores:
        row = movies.iloc[i]
        c_score = context_score(row, context)
        results.append((row['title'], sim, c_score))

    # 🔥 RE-RANK BASED ON CONTEXT
    results = sorted(results, key=lambda x: (x[2], x[1]), reverse=True)

    return results[:top_n]


print("\n🎬 Context-Aware Recommendations:\n")

results = recommend_with_context("Toy Story (1995)", context)

for i, (title, sim, c_score) in enumerate(results, 1):
    print(f"{i}. {title} (Similarity: {sim:.2f}, Context Score: {c_score})")

Q_table = {}


def get_max_q_for_user(user):
    user_qs = [v for (u, _), v in Q_table.items() if u == user]
    return max(user_qs) if user_qs else 0


def update_q(user, movie, reward, alpha=0.5, gamma=0.9):
    old_q = Q_table.get((user, movie), 0)

    future_q = get_max_q_for_user(user)

    new_q = old_q + alpha * (reward + gamma * future_q - old_q)

    Q_table[(user, movie)] = new_q


def get_q_score(user, movie):
    return Q_table.get((user, movie), 0)


def recommend_with_rl(movie_title, user_id, context, top_n=10):

    idx = movies[movies['title'] == movie_title].index

    if len(idx) == 0:
        return ["Movie not found!"]

    idx = idx[0]

    scores = list(enumerate(cos_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+15]

    results = []

    for i, sim in scores:
        row = movies.iloc[i]

        c_score = context_score(row, context)
        q_score = get_q_score(user_id, row['movieId'])

        results.append((row['title'], sim, c_score, q_score, row['movieId']))

    # 🔥 FINAL RANKING (RL + CONTEXT)
    results = sorted(results, key=lambda x: (0.5*x[3] + 0.3*x[2] + 0.2*x[1]),reverse =True)

    return results[:top_n]

from google.colab import drive
drive.mount('/content/drive')

def user_feedback(user_id, movie_id, liked=True):
    reward = 1 if liked else -1
    update_q(user_id, movie_id, reward)

Q_table = {}

user_id = 101

print("\n🎬 RL-Based Recommendations (Before Feedback):\n")

results = recommend_with_rl("Toy Story (1995)", user_id, context)

for i, (title, sim, c_score, q_score, mid) in enumerate(results, 1):
    print(f"{i}. {title} | Q={q_score:.2f} | Context={c_score}")


# Simulate user liking first 3 movies
for movie in results[:3]:
    user_feedback(user_id, movie[4], liked=True)


print("\n🎬 RL-Based Recommendations (After Feedback):\n")

results = recommend_with_rl("Toy Story (1995)", user_id, context)

for i, (title, sim, c_score, q_score, mid) in enumerate(results, 1):
    print(f"{i}. {title} | Q={q_score:.2f} | Context={c_score}")





🎬 Context-Aware Recommendations:

1. Toy Story 2 (1999) (Similarity: 0.80, Context Score: 2)
2. Goofy Movie, A (1995) (Similarity: 0.76, Context Score: 2)
3. Toy Story 3 (2010) (Similarity: 0.73, Context Score: 2)
4. Toys (1992) (Similarity: 0.70, Context Score: 2)
5. The Lego Movie (2014) (Similarity: 0.69, Context Score: 2)
6. Peanuts Movie, The (2015) (Similarity: 0.65, Context Score: 2)
7. Toy, The (1982) (Similarity: 0.64, Context Score: 2)
8. Hercules (1997) (Similarity: 0.63, Context Score: 2)
9. Super Mario Bros. (1993) (Similarity: 0.61, Context Score: 2)
10. Teenage Mutant Ninja Turtles III (1993) (Similarity: 0.60, Context Score: 2)
Mounted at /content/drive

🎬 RL-Based Recommendations (Before Feedback):

1. Toy Story 2 (1999) | Q=0.00 | Context=2
2. Goofy Movie, A (1995) | Q=0.00 | Context=2
3. Toy Story 3 (2010) | Q=0.00 | Context=2
4. Toys (1992) | Q=0.00 | Context=2
5. The Lego Movie (2014) | Q=0.00 | Context=2
6. Peanuts Movie, The (2015) | Q=0.00 | Context=2
7. Toy, T

In [5]:
%%writefile app.py

import streamlit as st
import pandas as pd
import requests
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline  # 🔥 NEW


st.set_page_config(page_title="Movie Recommender", layout="wide")

st.title("🎬 Smart Context-Aware Movie Recommender")

# ==============================
# 🎨 UI STYLING
# ==============================
st.markdown("""
<style>
.movie-card {
    background-color: #1e1e1e;
    padding: 15px;
    border-radius: 12px;
    margin-bottom: 15px;
}
.title {
    font-size: 18px;
    font-weight: bold;
}
</style>
""", unsafe_allow_html=True)

st.markdown("### 🎯 Get Personalized Recommendations")
st.caption("Upload your face image to detect emotion automatically")

# ==============================
# TMDB CONFIG
# ==============================
API_KEY = "c80d31b559df69fee4249b03dc001d90"

# ==============================
# CLEAN TITLE
# ==============================
def clean_title(title):
    return re.sub(r"\(\d{4}\)", "", title).strip()

def extract_year(title):
    match = re.search(r"\((\d{4})\)", title)
    return match.group(1) if match else ""

# ==============================
# TMDB FETCH
# ==============================
@st.cache_data(show_spinner=False)
def get_movie_details(title):
    clean = clean_title(title)
    year = extract_year(title)

    url = f"https://api.themoviedb.org/3/search/movie?api_key={API_KEY}&query={clean}&year={year}"

    try:
        response = requests.get(url).json()

        if response.get('results'):
            movie = max(response['results'], key=lambda x: x.get('vote_count', 0))

            poster_path = movie.get('poster_path')
            poster = f"https://image.tmdb.org/t/p/w500{poster_path}" if poster_path else None

            overview = movie.get('overview') or ""
            rating = movie.get('vote_average') or "N/A"

            return poster, overview, rating

    except:
        pass

    return None, "", "N/A"

# ==============================
# LOAD DATA
# ==============================
movies = pd.read_csv("ml-latest-small/movies.csv")
movies['overview'] = movies['title'].fillna('') + " " + movies['genres'].fillna('')

import numpy as np

# ==============================
# 🔥 LOAD EMBEDDINGS (BERT)
# ==============================
@st.cache_data
def load_embeddings():
    return np.load("embeddings.npy")

embeddings = load_embeddings()

# ==============================
# 🔥 TF-IDF (SECOND SIGNAL)
# ==============================
@st.cache_data
def compute_tfidf(data):
    vectorizer = TfidfVectorizer(stop_words='english')
    return vectorizer.fit_transform(data)

tfidf_matrix = compute_tfidf(movies['overview'])

# ==============================
# 🔥 HYBRID SIMILARITY
# ==============================
@st.cache_data
def compute_hybrid_similarity():
    sim_bert = cosine_similarity(embeddings)
    sim_tfidf = cosine_similarity(tfidf_matrix)

    # 🔥 FINAL HYBRID SCORE
    return 0.7 * sim_bert + 0.3 * sim_tfidf

cos_sim = compute_hybrid_similarity()

# ==============================
# RL MEMORY
# ==============================
if "Q_table" not in st.session_state:
    st.session_state.Q_table = {}

def get_q(user, movie):
    return st.session_state.Q_table.get((user, movie), 0)

def update_q(user, movie, reward):
    old = get_q(user, movie)
    new = old + 0.7 * (reward - old)
    st.session_state.Q_table[(user, movie)] = new

def encode_state(sim, c_score, q_score):
    return torch.tensor([sim, c_score, q_score], dtype=torch.float32)

# ==============================
# 🤖 DQN MODEL (PHASE 7)
# ==============================
class DQN(nn.Module):
    def __init__(self):
        super(DQN, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


dqn = DQN()
optimizer = optim.Adam(dqn.parameters(), lr=0.001)
loss_fn = nn.MSELoss()


# ==============================
# CONTEXT SCORING
# ==============================
def context_score(row, mood, time):
    score = 0
    genres = str(row['genres'])

    # 🌙 NIGHT
    if time == "night":
        if "Horror" in genres or "Thriller" in genres:
            score += 2

    # ☀️ DAY (NEW)
    if time == "day":
        if "Comedy" in genres or "Animation" in genres or "Adventure" in genres:
            score += 2

    # 😊 MOOD
    if mood == "happy":
        if "Comedy" in genres:
            score += 2
        elif "Adventure" in genres:
            score += 1

    if mood == "sad":
        if "Drama" in genres:
            score += 2
        elif "Romance" in genres:
            score += 1

    return score



# ==============================
# LOAD MODELS (CACHED)
# ==============================
@st.cache_resource
def load_hf_model():
    return pipeline(
        "image-classification",
        model="dima806/facial_emotions_image_detection"
    )

@st.cache_resource
def load_fer_model():
    try:
        from fer import FER
        return FER(mtcnn=False)
    except Exception as e:
        st.warning("FER model failed to load, using fallback")
        return None

hf_model = load_hf_model()
fer_model = load_fer_model()

# ==============================
# HYBRID EMOTION DETECTION 🔥
# ==============================
def detect_emotion(image):
    try:
        # HuggingFace (always works)
        hf_result = hf_model(image)
        hf_emotions = {r['label'].lower(): r['score'] for r in hf_result}

        # FER (optional)
        fer_emotions = {}

        if fer_model is not None:
            fer_result = fer_model.detect_emotions(np.array(image))
            if fer_result:
                fer_emotions = fer_result[0]["emotions"]

        # Combine
        combined = {}

        for emo in ["happy", "sad", "angry", "surprise", "neutral"]:
            combined[emo] = (
                0.7 * hf_emotions.get(emo, 0) +
                0.3 * fer_emotions.get(emo, 0)
            )

        return max(combined, key=combined.get)

    except:
        return "neutral"

# ==============================
# MAPPING (UNCHANGED)
# ==============================
def map_emotion_to_mood(emotion):
    if emotion in ["happy", "surprise"]:
        return "happy"
    elif emotion in ["sad", "angry", "fear"]:
        return "sad"
    else:
        return "neutral"

# ==============================
# EXPLANATION
# ==============================
def generate_explanation(row, mood, time, q_score, sim):
    genres = str(row['genres'])
    reasons = []

    score = 0.5*q_score + 0.3*sim

    if score > 0.75:
        level = "🔥 Highly Recommended"
    elif score > 0.6:
        level = "⭐ Strong Match"
    elif score > 0.5:
        level = "👍 Good Choice"
    else:
        level = "🔍 Worth Exploring"

    if sim > 0.7:
        reasons.append("very similar to your selected movie")
    elif sim > 0.5:
        reasons.append("shares similar themes")
    else:
        reasons.append("offers a different experience")

    if mood == "happy" and "Comedy" in genres:
        reasons.append("fits your happy mood")
    if mood == "sad" and "Drama" in genres:
        reasons.append("matches your mood")
    # ☀️ DAY (NEW)
    if time == "night":
        if "Horror" in genres:
            reasons.append("perfect for a night thrill")
        elif "Thriller" in genres:
            reasons.append("keeps you engaged at night")
    # ☀️ DAY (NEW)
    if time == "day":
        if "Comedy" in genres:
            reasons.append("great for a light daytime watch")
        elif "Animation" in genres:
            reasons.append("fun and refreshing for daytime")
        elif "Adventure" in genres:
            reasons.append("keeps the energy up during the day")

    # 🧠 RL reasoning (better)
    if q_score > 0.7:
        reasons.append("you strongly preferred similar movies before")
    elif q_score > 0.3:
        reasons.append("you showed interest in similar movies")

    # ✅ Ensure at least 1 reason exists
    if not reasons:
        reasons.append("it matches your overall preferences")

    return f"{level} 💡 Because it " + ", ".join(reasons[:2]) + "."

# ==============================
# TEXT HANDLING
# ==============================
def smart_truncate(text, length=150):
    if not text:
        return "No description available"
    if len(text) <= length:
        return text
    return text[:length].rsplit(" ", 1)[0] + "..."

def enhance_description(overview, title, genres):
    if not overview or len(overview) < 80:
        return f"{title} is a {genres} movie recommended based on your preferences."
    return overview

# ==============================
# RECOMMENDER
# ==============================
def recommend(movie_title, mood, time, user_id=1):
    idx = movies[movies['title'] == movie_title].index[0]

    scores = list(enumerate(cos_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:20]

    results = []

    for i, sim in scores:
        row = movies.iloc[i]
        c_score = context_score(row, mood, time)
        q_score = get_q(user_id, row['movieId'])

        state = encode_state(sim, c_score, q_score)

        with torch.no_grad():
            dqn_score = dqn(state).item()

        results.append((row, row['title'], sim, c_score, q_score, dqn_score, row['movieId']))

    results = sorted(results, key=lambda x: (0.5*x[5] + 0.25*x[4] + 0.15*x[3] + 0.10*x[2]), reverse=True)

    return results[:10]

# ==============================
# UI INPUT
# ==============================
col1, col2, col3 = st.columns(3)

with col1:
    movie = st.selectbox("🎥 Select Movie", movies['title'])

with col2:
    uploaded_file = st.file_uploader("📸 Upload Face Image", type=["jpg", "png", "jpeg"])

    if uploaded_file:
        image = Image.open(uploaded_file).convert("RGB")  # 🔥 FIX
        st.image(image, width=120)

        emotion = detect_emotion(image)
        mood = map_emotion_to_mood(emotion)

        st.success(f"Detected Emotion: {emotion}")
    else:
        mood = "neutral"

with col3:
    time = st.selectbox("⏰ Time", ["day", "night"])

# ==============================
# BUTTON
# ==============================
if st.button("🚀 Get Recommendations"):

    recs = recommend(movie, mood, time)

    st.subheader("🎯 Recommended Movies")

    for i, (row, title, sim, c_score, _, dqn_score, mid) in enumerate(recs, 1):

        q_score = get_q(1, mid)



        poster, overview, rating = get_movie_details(title)
        overview = enhance_description(overview, title, row['genres'])
        explanation = generate_explanation(row, mood, time, q_score, sim)

        st.markdown('<div class="movie-card">', unsafe_allow_html=True)

        col1, col2 = st.columns([1,3])

        with col1:
            if poster:
                st.image(poster)
            else:
                st.write("🎬 No Image")

        with col2:
            st.markdown(f'<div class="title">{i}. {title}</div>', unsafe_allow_html=True)
            st.caption(f"⭐ {rating} | 🎯 Context: {c_score} | 🧠 Learned score: {round(q_score,2)}")


            st.caption(f"🤖 AI Score: {round(dqn_score,2)}")

            preview = smart_truncate(overview)
            st.write(preview)

            with st.expander("📖 Read Full Description"):
                st.write(overview)

            st.info(explanation)

            col_like, col_dislike = st.columns(2)

            with col_like:


                if st.button(f"👍 Like {i}", key=f"like_{i}"):

                    # 🔁 Update Q-table
                    update_q(1, mid, 1)
                    new_q = get_q(1, mid)

                    # 🔥 TRAIN DQN
                    state = encode_state(sim, c_score, new_q).unsqueeze(0)

                    target = torch.tensor([[1.0]])  # reward = 1

                    pred = dqn(state)

                    loss = loss_fn(pred, target)

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    # 🔥 FORCE REFRESH
                    st.rerun()

            with col_dislike:
                if st.button(f"👎 Dislike {i}", key=f"dislike_{i}"):

                    update_q(1, mid, -1)
                    new_q = get_q(1, mid)

                    # 🔥 TRAIN DQN (negative reward)
                    state = encode_state(sim, c_score, new_q).unsqueeze(0)

                    target = torch.tensor([[-1.0]])

                    pred = dqn(state)

                    loss = loss_fn(pred, target)

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    # 🔥 FORCE REFRESH
                    st.rerun()

        st.markdown('</div>', unsafe_allow_html=True)



Writing app.py


In [6]:
!curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
!echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list
!sudo apt update
!sudo apt install ngrok

deb https://ngrok-agent.s3.amazonaws.com buster main
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://ngrok-agent.s3.amazonaws.com buster InRelease [20.3 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,004 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Hit:13 https://ppa.launchpadcon

In [7]:
!ngrok config add-authtoken 3B6s57mNS4Jf8p7S6eFwlBOmMBR_6EoqVsT5qDqUNyeTMtQBS

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [8]:
!python -m streamlit run app.py --server.port 8501 &



2026-05-11 04:08:35.039 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.231.228.66:8501

  Stopping...


In [9]:
!python -m streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > logs.txt 2>&1 &

In [10]:
!cat logs.txt

In [11]:
!lsof -i:8501

In [ ]:
!ngrok http 8501 --log=stdout

INFO[05-11|04:09:58] no configuration paths supplied 
INFO[05-11|04:09:58] using configuration at default config path path=/root/.config/ngrok/ngrok.yml
INFO[05-11|04:09:58] open config file                         path=/root/.config/ngrok/ngrok.yml err=nil
t=2026-05-11T04:09:58+0000 lvl=info msg="FIPS 140 mode" enabled=false
t=2026-05-11T04:09:58+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
t=2026-05-11T04:09:58+0000 lvl=info msg="client session established" obj=tunnels.session
t=2026-05-11T04:09:58+0000 lvl=info msg="tunnel session started" obj=tunnels.session
t=2026-05-11T04:09:59+0000 lvl=info msg="started tunnel" obj=tunnels name=command_line addr=http://localhost:8501 url=https://jovan-harmonizable-julie.ngrok-free.dev
t=2026-05-11T04:10:11+0000 lvl=info msg="join connections" obj=join id=990a5f514d55 l=127.0.0.1:8501 r=[2401:4900:97ec:dc63:d8a6:b5e2:4db8:40a]:62655
t=2026-05-11T04:10:12+0000 lvl=info msg="join connections" obj=join id=f4e85